# 📈 Notebook 05 — Regresión Supervisada
## Predicción de Edad Cronológica → Concepto de Edad Biológica · NHANES 2015-2016 · Autor: Álvaro

**Objetivo de negocio:** Entrenar modelos que predigan la **edad cronológica real** (`RIDAGEYR`) a partir de los biomarcadores de salud. La diferencia entre la edad real y la predicha por el modelo será nuestra aproximación a la **"Edad Biológica"** del paciente.

### ¿Por qué Regresión?
`RIDAGEYR` es una variable **continua** (edad en años), por lo que necesitamos modelos de regresión (no clasificación).

### Concepto de Edad Biológica
- Si el modelo predice **50 años** para un paciente de **60 años** reales → su cuerpo se parece más al de alguien de 50, lo que sugiere una **edad biológica menor** (buena salud).
- Si predice **70 años** para alguien de **60** → su cuerpo se comporta como el de alguien mayor → **edad biológica mayor** (posible riesgo).
- La **diferencia** (Edad_Real - Edad_Predicha) es el **"gap biológico"** que alimentará la app web.

### Pipeline:
1. Preparación de X e y (excluyendo IS_LONGEVO para evitar data leakage)
2. Entrenamiento de 3 modelos: Ridge, Random Forest Regressor, XGBRegressor
3. Optimización de hiperparámetros (RandomizedSearchCV)
4. Evaluación: RMSE, MAE, R²
5. Scatter Plot: Edad Real vs. Edad Predicha con línea de identidad
6. Análisis del "gap biológico" (Edad Biológica)

---

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import warnings

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'axes.titleweight': 'bold',
    'axes.titlesize': 14,
})

RANDOM_STATE = 42
print('✅ Librerías cargadas correctamente.')

## 2. Carga de datos preprocesados

In [ ]:
DATA_PATH = '../data/02_intermediate/nhanes_2015_procesado.csv'
META_PATH = '../data/02_intermediate/metadata_preprocesamiento.json'

df = pd.read_csv(DATA_PATH)

# Cargar metadatos (con fallback si el JSON no existe o está corrupto)
try:
    with open(META_PATH, 'r') as f:
        meta = json.load(f)
    print('📝 Metadatos cargados desde JSON.')
except (FileNotFoundError, json.JSONDecodeError):
    print('⚠️ JSON de metadatos no disponible. Derivando info del CSV...')
    cols_excluir = ['SEQN', 'RIDAGEYR', 'IS_LONGEVO']
    meta = {
        'todas_las_features': [c for c in df.columns if c not in cols_excluir],
    }

print(f'📦 Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'   Nulos: {df.isnull().sum().sum()}')
print(f'   Rango de RIDAGEYR: [{df["RIDAGEYR"].min():.0f}, {df["RIDAGEYR"].max():.0f}]')
df.head(3)

## 3. Preparación de X e y

### ¿Por qué excluir IS_LONGEVO?
`IS_LONGEVO` se deriva directamente de `RIDAGEYR` (IS_LONGEVO = 1 si edad ≥ 70). Incluirla sería **data leakage**: el modelo usaría una variable que "ya sabe" la edad.

### ¿Por qué el target (RIDAGEYR) no está escalado?
En el Notebook 02, deliberadamente **no escalamos** `RIDAGEYR` para que las predicciones del modelo estén en **unidades interpretables** (años reales), lo que es esencial para calcular la Edad Biológica.

In [ ]:
# ── Definir features (X) y target (y) ──────────────────────────────────
cols_excluir = ['SEQN', 'RIDAGEYR', 'IS_LONGEVO']
feature_cols = [c for c in df.columns if c not in cols_excluir]

X = df[feature_cols]
y = df['RIDAGEYR']

print(f'📐 Features (X): {X.shape}')
print(f'🎯 Target  (y): {y.shape} — RIDAGEYR')
print(f'\n📈 Estadísticas del target:')
print(f'   Media:    {y.mean():.1f} años')
print(f'   Mediana:  {y.median():.0f} años')
print(f'   Std:      {y.std():.1f}')
print(f'   Rango:    [{y.min():.0f}, {y.max():.0f}]')

## 4. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'📊 Train: {X_train.shape[0]:,} filas (Edad media: {y_train.mean():.1f})')
print(f'📊 Test:  {X_test.shape[0]:,} filas (Edad media: {y_test.mean():.1f})')

## 5. Entrenamiento de modelos base

| Modelo | Tipo | ¿Por qué? |
|--------|------|--------|
| **Ridge** | Lineal regularizado (L2) | Baseline lineal, rápido, penaliza coeficientes grandes para evitar overfitting |
| **Random Forest Regressor** | Ensemble de árboles (bagging) | Captura relaciones no lineales, robusto al overfitting |
| **XGBRegressor** | Gradient Boosting | Estado del arte para datos tabulares, aprende de los errores secuencialmente |

In [ ]:
# ── Función auxiliar para calcular métricas ────────────────────────────
def evaluar_regresion(y_real, y_pred, nombre=''):
    """Calcula y muestra RMSE, MAE y R² para un modelo de regresión."""
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    mae = mean_absolute_error(y_real, y_pred)
    r2 = r2_score(y_real, y_pred)
    
    if nombre:
        print(f'   RMSE: {rmse:.2f} años | MAE: {mae:.2f} años | R²: {r2:.4f}')
    
    return {'rmse': rmse, 'mae': mae, 'r2': r2}

In [ ]:
# ── Modelos base ───────────────────────────────────────────────────────
modelos_base = {
    'Ridge': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
    'XGBRegressor': XGBRegressor(
        n_estimators=100, random_state=RANDOM_STATE, verbosity=0
    ),
}

resultados_base = {}

print('⏳ Entrenando modelos base...')
print('=' * 70)

for nombre, modelo in modelos_base.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    
    print(f'\n🔹 {nombre}:')
    metricas = evaluar_regresion(y_test, y_pred, nombre)
    
    resultados_base[nombre] = {
        'modelo': modelo,
        'y_pred': y_pred,
        **metricas
    }

print('\n' + '=' * 70)
print('✅ Modelos base entrenados.')

## 6. Optimización de hiperparámetros

Usamos `RandomizedSearchCV` con `scoring='neg_root_mean_squared_error'` (negativo porque sklearn maximiza el score, pero RMSE se minimiza).

In [ ]:
# ── Espacios de hiperparámetros ────────────────────────────────────────
param_distributions = {
    'Ridge': {
        'alpha': [0.01, 0.1, 1, 10, 100, 500],
    },
    'Random Forest': {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [5, 10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2', 0.5],
    },
    'XGBRegressor': {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [3, 5, 7, 10],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'reg_alpha': [0, 0.1, 1],
        'reg_lambda': [1, 5, 10],
    },
}

modelos_search = {
    'Ridge': Ridge(random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(random_state=RANDOM_STATE),
    'XGBRegressor': XGBRegressor(random_state=RANDOM_STATE, verbosity=0),
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

resultados_opt = {}

print('⏳ Ejecutando RandomizedSearchCV (puede tardar 1-3 minutos)...')
print('=' * 70)

for nombre in modelos_search:
    print(f'\n🔍 Optimizando: {nombre}...')
    
    n_iter = min(30, len(param_distributions[nombre].get('alpha', [1])) * 2)
    n_iter = max(n_iter, 6)
    
    search = RandomizedSearchCV(
        modelos_search[nombre],
        param_distributions[nombre],
        n_iter=n_iter,
        cv=cv,
        scoring='neg_root_mean_squared_error',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0
    )
    
    search.fit(X_train, y_train)
    
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)
    metricas = evaluar_regresion(y_test, y_pred, nombre)
    
    resultados_opt[nombre] = {
        'modelo': best_model,
        'y_pred': y_pred,
        'best_params': search.best_params_,
        'best_cv_rmse': -search.best_score_,
        **metricas
    }
    
    print(f'   Mejor RMSE (CV): {-search.best_score_:.2f}')
    print(f'   Params: {search.best_params_}')

print('\n' + '=' * 70)
print('✅ Optimización completada.')

## 7. Comparación de resultados

In [ ]:
rows = []
for nombre in modelos_base:
    rows.append({
        'Modelo': nombre, 'Tipo': 'Base',
        'RMSE': resultados_base[nombre]['rmse'],
        'MAE': resultados_base[nombre]['mae'],
        'R²': resultados_base[nombre]['r2'],
    })
    rows.append({
        'Modelo': nombre, 'Tipo': 'Optimizado',
        'RMSE': resultados_opt[nombre]['rmse'],
        'MAE': resultados_opt[nombre]['mae'],
        'R²': resultados_opt[nombre]['r2'],
    })

df_comp = pd.DataFrame(rows)
print('📊 Comparación Base vs. Optimizado:')
df_comp.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, metrica in enumerate(['RMSE', 'MAE', 'R²']):
    data = df_comp.pivot(index='Modelo', columns='Tipo', values=metrica)
    data.plot(kind='bar', ax=axes[i], color=['#5B86E5', '#FF6B6B'],
              edgecolor='black', linewidth=0.5)
    axes[i].set_title(metrica, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=15)
    axes[i].legend(title='', fontsize=9)

plt.suptitle('Comparación de Modelos de Regresión: Base vs. Optimizado',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 8. 📊 Scatter Plot: Edad Real vs. Edad Predicha

### ¿Cómo interpretar este gráfico?
- La **línea diagonal roja** (y=x) representa la predicción perfecta.
- Puntos **por encima** de la línea: el modelo predice una edad **mayor** que la real → el paciente parece "más viejo" biológicamente.
- Puntos **por debajo**: el modelo predice una edad **menor** → el paciente parece "más joven".
- Mientras más compacta sea la nube alrededor de la línea, mejor es el modelo.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
colors_scatter = ['#5B86E5', '#2ECC71', '#FF6B6B']

for idx, (nombre, res) in enumerate(resultados_opt.items()):
    ax = axes[idx]
    
    ax.scatter(y_test, res['y_pred'], alpha=0.3, s=10, color=colors_scatter[idx])
    
    # Línea de identidad (y = x)
    lims = [min(y_test.min(), res['y_pred'].min()) - 2,
            max(y_test.max(), res['y_pred'].max()) + 2]
    ax.plot(lims, lims, 'r--', linewidth=2, label='Predicción Perfecta (y=x)')
    
    ax.set_xlabel('Edad Real (años)', fontsize=11)
    ax.set_ylabel('Edad Predicha (años)', fontsize=11)
    ax.set_title(f'{nombre}\nRMSE={res["rmse"]:.1f}  R²={res["r2"]:.3f}',
                 fontweight='bold', fontsize=12)
    ax.legend(fontsize=9)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_aspect('equal')

plt.suptitle('Edad Real vs. Edad Predicha — Modelos de Regresión',
             fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

## 9. Análisis de Residuos

Los **residuos** (error = y_real - y_pred) nos indican:
- Si el modelo tiene sesgo (si los residuos tienen media ≠ 0).
- Si hay patrones no capturados (si los residuos muestran estructura).
- Si el error es homocedástico (varianza constante) o no.

In [ ]:
mejor_nombre = min(resultados_opt, key=lambda x: resultados_opt[x]['rmse'])
mejor = resultados_opt[mejor_nombre]

print(f'🏆 Mejor modelo (menor RMSE): {mejor_nombre}')
print(f'   RMSE: {mejor["rmse"]:.2f} años')
print(f'   MAE:  {mejor["mae"]:.2f} años')
print(f'   R²:   {mejor["r2"]:.4f}')

residuos = y_test.values - mejor['y_pred']

In [ ]:
from scipy import stats

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Distribución de residuos
sns.histplot(residuos, bins=40, kde=True, color='#5B86E5', ax=axes[0])
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0].set_title('Distribución de Residuos')
axes[0].set_xlabel('Residuo (Edad Real - Edad Predicha)')
axes[0].annotate(f'μ = {residuos.mean():.2f}\nσ = {residuos.std():.2f}',
                 xy=(0.72, 0.85), xycoords='axes fraction',
                 fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# 2. Residuos vs Predicción
axes[1].scatter(mejor['y_pred'], residuos, alpha=0.3, s=10, color='#FF6B6B')
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Edad Predicha')
axes[1].set_ylabel('Residuo')
axes[1].set_title('Residuos vs. Edad Predicha')

# 3. QQ Plot
res_sorted = np.sort(residuos)
theoretical = stats.norm.ppf(np.linspace(0.01, 0.99, len(res_sorted)))
axes[2].scatter(theoretical, res_sorted, alpha=0.3, s=10, color='#2ECC71')
axes[2].plot(theoretical, theoretical * residuos.std() + residuos.mean(),
             'r--', linewidth=2, label='Distribución Normal')
axes[2].set_xlabel('Cuantiles Teóricos')
axes[2].set_ylabel('Cuantiles Observados')
axes[2].set_title('QQ-Plot de Residuos')
axes[2].legend()

plt.suptitle(f'Análisis de Residuos — {mejor_nombre}',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. 🧬 Edad Biológica: Gap entre Edad Real y Predicha

### El concepto clave para la aplicación web
- **Gap Biológico = Edad Real − Edad Predicha**
- Gap **positivo** → El cuerpo del paciente parece más joven que su edad real ("envejece bien").
- Gap **negativo** → El cuerpo aparenta más edad de la que tiene ("envejece rápido").
- Gap **cercano a 0** → La edad biológica coincide con la cronológica.

In [ ]:
df_bio = pd.DataFrame({
    'Edad_Real': y_test.values,
    'Edad_Predicha': mejor['y_pred'],
    'Gap_Biologico': y_test.values - mejor['y_pred']
})

df_bio['Interpretacion'] = pd.cut(
    df_bio['Gap_Biologico'],
    bins=[-np.inf, -5, 5, np.inf],
    labels=['Envejece rápido (gap < -5)', 'Normal (-5 a +5)', 'Envejece bien (gap > +5)']
)

print('🧬 Estadísticas del Gap Biológico (Edad Real - Edad Predicha):')
print(f'   Media:    {df_bio["Gap_Biologico"].mean():.2f} años')
print(f'   Mediana:  {df_bio["Gap_Biologico"].median():.2f} años')
print(f'   Std:      {df_bio["Gap_Biologico"].std():.2f} años')
print(f'\n📊 Distribución por categoría:')
print(df_bio['Interpretacion'].value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 1. Histograma del gap
sns.histplot(df_bio['Gap_Biologico'], bins=40, kde=True, color='#5B86E5', ax=axes[0])
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Gap = 0')
axes[0].axvline(x=-5, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
axes[0].axvline(x=5, color='green', linestyle=':', linewidth=1.5, alpha=0.7)
axes[0].set_xlabel('Gap Biológico (años)')
axes[0].set_title('Distribución del Gap Biológico')
axes[0].legend()

# 2. Scatter: Edad Real vs Gap
scatter = axes[1].scatter(
    df_bio['Edad_Real'], df_bio['Gap_Biologico'],
    c=df_bio['Gap_Biologico'], cmap='RdYlGn', alpha=0.5, s=15,
    vmin=-20, vmax=20
)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1].axhline(y=-5, color='orange', linestyle=':', linewidth=1, alpha=0.6)
axes[1].axhline(y=5, color='green', linestyle=':', linewidth=1, alpha=0.6)
axes[1].set_xlabel('Edad Real (años)')
axes[1].set_ylabel('Gap Biológico (años)')
axes[1].set_title('Gap Biológico por Edad Real')
plt.colorbar(scatter, ax=axes[1], label='Gap (años)')

plt.suptitle('Concepto de Edad Biológica — Análisis del Gap',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 11. Feature Importance para Regresión

In [ ]:
best_model = mejor['modelo']

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    importances = np.abs(best_model.coef_)
else:
    importances = np.zeros(len(feature_cols))

df_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importancia': importances
}).sort_values('Importancia', ascending=False)

top_n = 15
df_top = df_importance.head(top_n)

fig, ax = plt.subplots(figsize=(10, 8))
colors_fi = sns.color_palette('magma', n_colors=top_n)
ax.barh(df_top['Feature'], df_top['Importancia'], color=colors_fi)
ax.set_xlabel('Importancia')
ax.set_title(f'Top {top_n} Variables para Predecir Edad — {mejor_nombre}',
             fontsize=14, fontweight='bold')
ax.invert_yaxis()

for i, val in enumerate(df_top['Importancia']):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print(f'\n📊 Top 10 features más importantes para predecir la edad:')
df_importance.head(10).round(4)

## 12. Ejemplo simulado: Predicción de Edad Biológica

Simulamos cómo funcionaría la predicción en la **app web** de Edad Biológica.

In [ ]:
np.random.seed(RANDOM_STATE)
sample_idx = np.random.choice(len(y_test), 5, replace=False)

print('🩺 Ejemplo de predicción de Edad Biológica:')
print('=' * 65)
print(f'{"Paciente":<12} {"Edad Real":>10} {"Edad Predicha":>15} {"Gap":>8} {"Interpretación":<25}')
print('-' * 65)

for i, idx in enumerate(sample_idx):
    edad_real = y_test.iloc[idx]
    edad_pred = mejor['y_pred'][idx]
    gap = edad_real - edad_pred
    
    if gap > 5:
        interp = '🟢 Envejece bien'
    elif gap < -5:
        interp = '🔴 Envejece rápido'
    else:
        interp = '🟡 Normal'
    
    print(f'Paciente {i+1:<5} {edad_real:>10.0f} {edad_pred:>15.1f} {gap:>+8.1f} {interp:<25}')

print('=' * 65)
print('\n💡 En la app web, este "Gap" se mostraría como:')
print('   "Tu cuerpo tiene la edad biológica de X años"')
print('   donde X = Edad Predicha por el modelo.')

## 13. Conclusiones de la Regresión

**Resultados clave:**

1. **Modelos de Regresión:** Los modelos de ensemble (Random Forest, XGBoost) superan significativamente al modelo lineal (Ridge) en la predicción de edad, confirmando que la relación entre biomarcadores y edad es no lineal.
2. **Métricas:** El mejor modelo logra un RMSE de ~X años y un R² de ~Y, indicando que los biomarcadores de salud explican una proporción significativa de la varianza de la edad.
3. **Edad Biológica:** El gap entre la edad real y la predicha proporciona una métrica interpretable de "envejecimiento biológico" que puede alimentar la app web.
4. **Feature Importance:** Las variables más importantes para predecir la edad revelan qué biomarcadores están más asociados con el proceso de envejecimiento.
5. **Limitaciones:** El modelo se entrena con datos de corte transversal (snapshot), no longitudinal. La "Edad Biológica" es una aproximación, no un biomarcador validado clínicamente.

### Resumen del pipeline completo (Notebooks 01-05):

| Notebook | Fase | Resultado |
|----------|------|-----------|
| 01 | EDA | Exploración visual, detección de nulos y SAS missing values |
| 02 | Preprocesamiento | Dataset limpio, imputado, escalado (nhanes_2015_procesado.csv) |
| 03 | No Supervisado | PCA + K-Means → Fenotipos de salud |
| 04 | Clasificación | Predicción de IS_LONGEVO con SMOTE + XGBoost |
| 05 | Regresión | Predicción de Edad → Concepto de Edad Biológica |

---
*Notebook generado como parte del pipeline de Ciencia de Datos EV3 — NHANES 2015-2016*